In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

"""
    第一步,文档切片
"""
with open('../day07/resources/评估.md','r',encoding='utf-8')  as file:
    doc = file.read()

# 创建markdown切分对象,做切片操作
markdown_splitter =  MarkdownHeaderTextSplitter(
    # 切分规则:按段落切分
    [
        # metadata包含三个数值
        # 会被作为表结构的schema作为字段
        ("#", "t1"),
        ("##", "t2"),
        ("###", "t3"),
    ]
)

# 文档切分
chunks = markdown_splitter.split_text(doc)

In [2]:
"""
    使用本地向量化模型
"""
from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

In [3]:
"""
    初始化向量数据库连接对象
"""
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings

bm25 = BM25BuiltInFunction(
    analyzer_params={"type": "chinese"},
    output_field_names="wcy_sparse"      # 这一行是关键！
)
vector_store = Milvus(
    embedding_function=embeddings,  # 稠密向量模型
    collection_name="xiao_wang_collection",  # collection名称
    builtin_function=bm25,
    vector_field=["wcy_dense", "wcy_sparse"],  # 向量字段，包括稠密和稀疏
    connection_args={
        "uri": settings.rag.milvus_url,  # milvus的uri路径
    },
    drop_old=True,  # 是否删除旧的collection，避免重复创建
    auto_id=True
)

In [4]:
# 直接新增文本
documents = [
    chunks[i:i+10]
    for i in range(0,len(chunks,),10)
]
for document in documents:
    res = vector_store.add_documents(document)